# Poker + Trading CFR Demo

This notebook is meant to be presentation-friendly. It opens the saved benchmark artifacts first, then gives you one-cell helpers to rerun the small README snapshot experiments.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = ROOT / "results"
VENV_PYTHON = ROOT / ".venv" / "bin" / "python"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")
print(f"Results dir: {RESULTS}")

## Load Saved Summaries

In [ ]:
def load_summary(name: str) -> dict:
    path = RESULTS / f"{name}_summary.json"
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def summary_markdown(summary: dict) -> str:
    lines = [
        f"### {summary['game'].title()} summary",
        "",
        f"- Evaluation: {summary['evaluation']}",
        f"- Iterations: {summary['iterations']}",
        f"- Eval every: {summary['eval_every']}",
        f"- Seeds: {summary['seeds']}",
        "",
        "| Method | Final exploitability mean | Final exploitability std | Self-play value mean |",
        "| --- | --- | --- | --- |",
    ]
    for label, trainer_summary in summary['trainers'].items():
        lines.append(
            f"| {label} | {trainer_summary['final_exploitability_mean']:.4f} | {trainer_summary['final_exploitability_std']:.4f} | {trainer_summary['self_play_value_mean']:.4f} |"
        )
    return "\n".join(lines)


poker_summary = load_summary("poker")
trading_summary = load_summary("trading")
display(Markdown(summary_markdown(poker_summary)))
display(Markdown(summary_markdown(trading_summary)))

## Show Saved Figures

In [ ]:
for title, filename in [
    ("Poker exploitability", "poker_exploitability.png"),
    ("Poker strategy", "poker_strategy.png"),
    ("Trading exploitability", "trading_exploitability.png"),
    ("Trading strategy", "trading_strategy.png"),
]:
    display(Markdown(f"### {title}"))
    display(Image(filename=str(RESULTS / filename)))

## Optional: rerun the README snapshot experiments

These use the same small budgets that produced the saved summaries in `results/`.

In [ ]:
def run_snapshot(script_name: str, env_updates: dict) -> None:
    env = os.environ.copy()
    env.update(env_updates)
    subprocess.run([str(VENV_PYTHON), script_name], cwd=ROOT, env=env, check=True)


run_snapshot(
    "run_poker.py",
    {
        "POKER_ITERATIONS": "200",
        "POKER_EVAL_EVERY": "50",
        "POKER_EVAL_EPISODES": "400",
    },
)

run_snapshot(
    "run_trading.py",
    {
        "TRADING_ITERATIONS": "120",
        "TRADING_EVAL_EVERY": "30",
        "TRADING_EVAL_EPISODES": "120",
        "TRADING_HORIZON": "2",
        "TRADING_SCENARIOS": "128",
    },
)

print("Snapshot runs completed.")

## Reload artifacts after rerunning

In [ ]:
poker_summary = load_summary("poker")
trading_summary = load_summary("trading")
display(Markdown(summary_markdown(poker_summary)))
display(Markdown(summary_markdown(trading_summary)))